# N073 · 二叉搜索树与有序操作

**目标：** 利用全子树范围约束而非只比较父子。

**先修：** N069, N071, N042。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** BST不变量；中序有序；上下界；搜索/插入/删除；后继；退化。

## 从问题到算法

BST不变量是整棵左子树小于根、整棵右子树大于根，不只是父子局部比较。中序应严格递增。本章采用唯一键集合语义，重复插入不创建新节点。删除有两个孩子的节点时，用右子树最小键替换，再删除该后继节点；这会修改原节点值，接口不承诺保留被删除键所在对象。

## 最小实现

**本章接口：** `is_valid_bst(root)`、`kth_smallest(root, k)`、`insert_bst(root, x)`、`delete_bst(root, x)`

In [1]:
class TreeNode:
    def __init__(self,val=0,left=None,right=None):
        self.val,self.left,self.right=val,left,right

def tree_from_level(values):
    """Construct a finite binary tree from level order with None placeholders."""
    from collections import deque
    if not values or values[0] is None: return None
    root=TreeNode(values[0]); queue=deque([root]); i=1
    while queue and i<len(values):
        node=queue.popleft()
        if values[i] is not None: node.left=TreeNode(values[i]); queue.append(node.left)
        i+=1
        if i<len(values):
            if values[i] is not None: node.right=TreeNode(values[i]); queue.append(node.right)
            i+=1
    return root

def _inorder_nodes(root):
    stack=[]; node=root
    while stack or node:
        while node: stack.append(node); node=node.left
        node=stack.pop(); yield node; node=node.right

def is_valid_bst(root):
    previous=None; first=True
    for node in _inorder_nodes(root):
        if not first and node.val<=previous: return False
        previous=node.val; first=False
    return True

def kth_smallest(root,k):
    if k<1: raise ValueError('positive rank required')
    for i,node in enumerate(_inorder_nodes(root),1):
        if i==k: return node.val
    raise ValueError('k exceeds number of nodes')

def insert_bst(root,x):
    if root is None: return TreeNode(x)
    node=root
    while True:
        if x==node.val: return root
        side='left' if x<node.val else 'right'
        child=getattr(node,side)
        if child is None: setattr(node,side,TreeNode(x)); return root
        node=child

def delete_bst(root,x):
    parent=None; node=root
    while node and node.val!=x:
        parent=node; node=node.left if x<node.val else node.right
    if not node: return root
    if node.left and node.right:
        successor_parent=node; successor=node.right
        while successor.left: successor_parent=successor; successor=successor.left
        node.val=successor.val; parent,node=successor_parent,successor
    child=node.left if node.left else node.right
    if parent is None: return child
    if parent.left is node: parent.left=child
    else: parent.right=child
    return root

## 正确性、前提与复杂度

中序严格递增等价于唯一键BST的全局区间约束。右子树最小值大于全部左子树值且不大于剩余右子树值，替换后仍维持BST次序。

**代价：** 查询插删O(h)，最坏O(n)，不能当成平衡树O(log n)；验证O(n)，第k小O(h+k)，遍历栈O(h)。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

r=None; rows=[]
for x in [5,3,6,2,4,7]:
    r=insert_bst(r,x); rows.append((f'插入{x}',[n.val for n in _inorder_nodes(r)]))
r=delete_bst(r,5); rows.append(('删除根5',[n.val for n in _inorder_nodes(r)]))
show_table(['操作','中序'],rows)

操作,中序
插入5,[5]
插入3,"[3, 5]"
插入6,"[3, 5, 6]"
插入2,"[2, 3, 5, 6]"
插入4,"[2, 3, 4, 5, 6]"
插入7,"[2, 3, 4, 5, 6, 7]"
删除根5,"[2, 3, 4, 6, 7]"


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
assert not is_valid_bst(tree_from_level([5,1,4,None,None,3,6]))
from random import Random
rng=Random(73); r=None; ref=set()
for _ in range(200):
    x=rng.randrange(30)
    if rng.randrange(2): r=insert_bst(r,x); ref.add(x)
    else: r=delete_bst(r,x); ref.discard(x)
    assert is_valid_bst(r) and [n.val for n in _inorder_nodes(r)]==sorted(ref)
    for k,x in enumerate(sorted(ref),1): assert kth_smallest(r,k)==x
print('N073: 所有本章断言通过')

N073: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 明确重复键策略。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 解释普通BST不保证对数高度。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 98. Validate Binary Search Tree（canonical）
- 230. Kth Smallest Element in a BST（canonical）
- 450. Delete Node in a BST（canonical）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。